# DU-Net (Ye et al., 2022)

A densely connected U-Net trained with an L2 loss on the 22,000 training patches to map 800 m upward-continued data to the
ground-level field. Weights: `weights/dunet.pth`.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward in a single step (zero padding to 36 × 36).

In [2]:
from gravdc import upward_continue

names, x = load_test_set(DATA_DIR)
x = x.to(device)
y = upward_continue(x, 800)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference

In [3]:
model = load_model("dunet", device)
with torch.no_grad():
    x_hat = model(y)

## Metrics

In [4]:
per_image = evaluate(x, x_hat)
compare("DU-Net", summarize(per_image))
save_result("DU-Net", per_image, names, RESULTS_DIR)

DU-Net
  metric             this run              Table 3
  PSNR       36.2222 ± 1.8690     36.2222 ± 1.8690  ✓
  SSIM        0.9861 ± 0.0088      0.9861 ± 0.0088  ✓
  RE          0.0768 ± 0.0144      0.0768 ± 0.0144  ✓
  RMSE        0.0270 ± 0.0073      0.0270 ± 0.0073  ✓


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"DU-Net": x_hat[k]}, title=names[k])